# Maximum Likelihood Mapmaking

This notebook demonstrates running the M.L. Mapmaker.  We will work with with a whole wafer-observation, but only use a few cores.  So the execution of the notebook will be **much** slower than processing the same data in the batch job.  See the slurm script in this directory for a batch-processing example.  NOTE:  Original example notebook is in the [sotask git repo](https://github.com/simonsobs/sotask/examples).

### Assumptions

- This notebook is designed to run at jupyter.nersc.gov or a system where data is available

## Environment and Imports

There are several runtime options in toast which can be controlled via environment variables.  Here we could change the logging level to DEBUG, which would be fine since we are only running on one process with a small amount of data.  These need to be set at the beginning before other packages are imported.

In [1]:
import os
os.environ["TOAST_LOGLEVEL"] = "INFO"
# This is needed before importing toast, and should
# match the value passed to the '-t' option of %toast
os.environ["OMP_NUM_THREADS"] = "4"

In [2]:
# TOAST interactive startup
import toast.interactive
%load_ext toast.interactive
%toast -p 1 -t 4 -a

TOAST INFO: Using 1 processes with 4 threads each.


In [3]:
# General Imports
import os
import re
import glob
import argparse

import numpy as np
import astropy.io.fits as af
from astropy import units as u
import matplotlib.pyplot as plt

# Import sotodlib.toast first, since that sets default object names
# to use in toast.
import sotodlib.toast as sotoast

import toast
from toast.observation import default_values as defaults

import sotodlib
import sotodlib.toast.ops as so_ops

abspath = os.path.dirname(os.path.abspath("sim_ml_mapmaking.ipynb"))

In [4]:
# MPI communicator
world, procs, rank = toast.mpi.get_world()

## Configuration and Data Selection

Here we choose one wafer of one observation from satp3.  The detector selection can be expressed as a dictionary compatible with the syntax accepted by `Context.get_obs()`.

In [5]:
# The observation(s) we are using
tele_name = "lat"

# Standard context
context_file = f"/global/cfs/cdirs/sobs/metadata/{tele_name}/contexts/use_this_local.yaml"
context_file = f"/cephfs/soukdata/data/tracked/metadata/{tele_name}/contexts/use_this_local.yaml"
if not os.path.exists(context_file):
    raise FileNotFoundError("The context file was not found, please make sure you have specified the correct file for this local machine.")

# Observation for this exercise
obs_id = "obs_1761519423_lati1_111"

# Select one wafer / frequency
det_select = {"wafer.bandpass":"f090","stream_id":"ufm_mv21"}

# Config file location
config_dir = f"{abspath}/configs"
config_lat_dir = os.path.join(config_dir, "lat")

if not os.path.exists(config_lat_dir):
    raise FileNotFoundError(f"{config_lat_dir}")
# Optional site-pipeline preprocessing config.  These are specialized
# operations that do not yet have equivalents in toast.  These operations
# will be applied when the data is loaded.
site_preproc = os.path.join(config_lat_dir, "site_proc.yml")

# Area file.  This defines the projection for the ML mapmaker.
area_file = "so_geometry_v20241015_2p0.fits"

In [6]:
# Output directory.
out_dir = "."
if rank == 0:
    os.makedirs(out_dir, exist_ok=True)
if world is not None:
    out_dir = world.bcast(out_dir, root=0)

## Helper Functions

These are copied from the `sotask` package for these stand-alone versions of the notebooks.

In [7]:
def group_dets(obs, mask=defaults.det_mask_nonscience):
    """Organize timestreams associated with detectors."""
    pat = re.compile(r"(demod[024ri]+)_(.*)")
    fp = obs.telescope.focalplane.detector_data
    if "pixel" in fp.colnames:
        det_to_pix = dict()
        for row in fp:
            det_to_pix[row["name"]] = row["pixel"]
    else:
        det_to_pix = {x: x for x in fp["name"]}
    groups = dict()
    
    for det in obs.select_local_detectors(flagmask=mask):
        pix = det_to_pix[det]
        if pix not in groups:
            groups[pix] = dict()
        mat = pat.match(det)
        if mat is None:
            # We have normal, undemodulated data
            det_type = "normal"
            det_name = det
        else:
            # This is a demod TOD
            det_type = mat.group(1)
            det_name = mat.group(2)
        if det_name not in groups[pix]:
            groups[pix][det_name] = dict()
            groups[pix][det_name]["normal"] = det_name
        groups[pix][det_name][det_type] = det
    return groups

def plot_obs_dets(
    obs,
    d_start=0,
    d_end=None,
    s_start=0,
    s_end=None,
    view=None,
    signal=defaults.det_data,
    mask=defaults.det_mask_nonscience,
    file=None,
    pattern=None,
):
    """Plot some unflagged detectors in an observation.

    Args:
        obs (Observation):  The observation
        d_start (int):  The starting local detector index to plot.
        d_end (int): The local detector index limit to plot.
        s_start (int):  The starting sample index to plot.
        s_end (int):  The sample index limit to plot
        view (str):  The optional intervals to overplot.
        signal (str):  The detdata name to plot.
        mask (int):  Mask for selecting good dets.
        file (str):  If not None, save to file instead.
        pattern (str):  Regex pattern to select detector IDs

    """
    if s_start is None:
        s_start = 0
    if s_end is None:
        s_end = obs.n_local_samples
    slc = slice(s_start, s_end, 1)

    # Interval view
    rects = None
    if view is not None and view in obs.intervals:
        rects = list()
        for intr in obs.intervals[view]:
            begin = intr.first
            end = intr.last
            if begin < s_end and end > s_start:
                # Some overlap
                if begin < s_start:
                    begin = s_start
                if end > s_end:
                    end = s_end
                rects.append((begin, end))

    # Get valid dets
    groups = group_dets(obs, mask=mask)
    n_all_dets = np.sum([len(y) for x, y in groups.items()])

    dets = dict()
    if d_start is None:
        d_start = 0
    if d_end is None:
        d_end = n_all_dets
    cur = 0
    for grp, gdets in groups.items():
        ngdet = len(gdets)
        if cur + ngdet > d_start and cur < d_end:
            dets.update(gdets)
        cur += ngdet

    if pattern is not None:
        det_pat = re.compile(pattern)
    else:
        det_pat = re.compile(r".*")
    fp = obs.telescope.focalplane.detector_data
    fpvals = {x: y for x, y in zip(fp["det_info:readout_id"], fp["det_info:det_id"])}

    # Compute number of plots
    n_plot = 1
    for d, dtod in dets.items():
        if "demod2r" in dtod:
            # We have demodulated data with 2f component
            n_plot = 4
            demod = True
            demod2f = True
        elif "demod0" in dtod:
            # We have normal 0 and 4f demodulated data
            n_plot = 3
            demod = True
            demod2f = False
        else:
            # Normal data
            demod = False
            demod2f = False

    # Extra plot for flags
    n_plot += 1

    fig, axs = plt.subplots(nrows=n_plot, ncols=1, dpi=100, figsize=(12, 8 * n_plot))

    # Shared flags
    axs[-1].plot(
        obs.shared[defaults.times].data[slc],
        obs.shared[defaults.shared_flags].data[slc],
        "-",
        color="black",
        label="Shared Flags",
    )

    for idet, (det, dtod) in enumerate(dets.items()):
        if idet < d_start or idet >= d_end:
            continue
        iplot = 0
        if demod:
            if det_pat.match(fpvals[det]) is None:
                continue
            # Plot demod0
            axs[iplot].plot(
                obs.shared[defaults.times].data[slc],
                obs.detdata[signal][dtod["demod0"], slc],
                "-",
                label=det,
            )
            iplot += 1
            if demod2f:
                # Plot 2f magnitude
                d2r = dtod["demod2r"]
                d2i = dtod["demod2i"]
                d2data = np.sqrt(
                    obs.detdata[signal][d2r, slc] ** 2 +
                    obs.detdata[signal][d2i, slc] ** 2
                )
                axs[iplot].plot(
                    obs.shared[defaults.times].data[slc],
                    d2data,
                    "-",
                    label=det,
                )
                iplot += 1
            # Plot 4f
            axs[iplot].plot(
                obs.shared[defaults.times].data[slc],
                obs.detdata[signal][dtod["demod4r"], slc],
                "-",
                label=det,
            )
            iplot += 1
            axs[iplot].plot(
                obs.shared[defaults.times].data[slc],
                obs.detdata[signal][dtod["demod4i"], slc],
                "-",
                label=det,
            )
            iplot += 1
            axs[iplot].plot(
                obs.shared[defaults.times].data[slc],
                obs.detdata[defaults.det_flags][dtod["demod0"], slc],
                "-",
                label=det,
            )
        else:
            if det_pat.match(fpvals[det]) is None:
                continue
            # Plot normal TOD
            axs[iplot].plot(
                obs.shared[defaults.times].data[slc],
                obs.detdata[signal][dtod["normal"], slc],
                "-",
                label=det,
            )
            iplot += 1
            axs[iplot].plot(
                obs.shared[defaults.times].data[slc],
                obs.detdata[defaults.det_flags][dtod["normal"], slc],
                "-",
                label=det,
            )
            iplot += 1

    # Views and Labels
    trects = None
    if rects is not None:
        trects = [
            (obs.shared[defaults.times].data[x], obs.shared[defaults.times].data[y-1])
            for x, y in rects
        ]

    def _plot_rects(ax):
        if trects is None:
            return
        ymin, ymax = ax.get_ylim()
        for rct in trects:
            ax.add_patch(
                mpatches.Rectangle(
                    (rct[0], ymin), 
                    rct[1]-rct[0], 
                    ymax-ymin,
                    fill=True,
                    facecolor="gray",
                    alpha=0.2,
                )
            )

    iplot = 0
    if demod:
        _plot_rects(axs[iplot])
        axs[iplot].legend(loc="best")
        axs[iplot].set_title("Demodulated Intensity")
        iplot += 1
        if demod2f:
            _plot_rects(axs[iplot])
            axs[iplot].legend(loc="best")
            axs[iplot].set_title(r"Demodulated 2f Magnitude ($\sqrt{{demod2r}^2 + {demod2i}^2}$)")
            iplot += 1
        _plot_rects(axs[iplot])
        axs[iplot].legend(loc="best")
        axs[iplot].set_title("Demodulated Q")
        iplot += 1
        _plot_rects(axs[iplot])
        axs[iplot].legend(loc="best")
        axs[iplot].set_title("Demodulated U")
        iplot += 1
    else:
        _plot_rects(axs[iplot])
        axs[iplot].legend(loc="best")
        axs[iplot].set_title("Timestreams")
        iplot += 1
    _plot_rects(axs[iplot])
    axs[iplot].legend(loc="best")
    axs[iplot].set_title("Flags")
    iplot += 1

    if file is None:
        plt.show()
    else:
        fig.savefig(file)
    plt.close()

## Load Config Files

Instead of manually running each operator and specifying individual options, we use the same config files that we will use in the corresponding batch jobs.  We use some of the "nominal" configs for many steps and also load a local mapmaking config file in the same directory as this notebook.

In [8]:
parser = argparse.ArgumentParser()
opts = [
    "--config",
    os.path.join(config_dir, "load_context.yml"), # General data loading
    os.path.join(config_dir, "pointing_detector.yml"), # Detector quaternion pointing
    os.path.join(config_dir, "pre_proc.yml"), # Low-level preprocessing
    os.path.join(config_lat_dir, "pointing.yml"),
    os.path.join(config_dir, "sim_pointing_detector.yml"),
    os.path.join(config_lat_dir, "sim_pointing.yml"),
    os.path.join(config_dir, "sim_sky_alm.yml"),
    os.path.join(config_dir, "sim_noise.yml"),
    os.path.join(config_dir, "sim_atmosphere.yml"),
    os.path.join(config_dir, "lat", "simulation.yml"),
    os.path.join(config_dir, "lat", "mapmaking.yml"),
]

In [9]:
config, otherargs, runargs = toast.config.run_config(parser, opts=opts)
job = toast.traits.create_from_config(config)
job_ops = job.operators

In [13]:
print(type(job))
print(type(job_ops))
print(type(job_ops.load))
?sotodlib.toast.ops.load_context.LoadContext.apply

<class 'types.SimpleNamespace'>
<class 'types.SimpleNamespace'>
<class 'sotodlib.toast.ops.load_context.LoadContext'>


Signature:
sotodlib.toast.ops.load_context.LoadContext.apply(
    self,
    data,
    detectors=None,
    **kwargs,
)
Docstring:
Run exec() and finalize().

This is a convenience wrapper that calls exec() exactly once with an optional
detector list and then immediately calls finalize().  This is really only
useful when working interactively to save a bit of typing.  When a `Pipeline`
is calling other operators it will always use exec() and finalize() explicitly.

After calling this, any future calls to exec() may produce unexpected results,
since finalize() has already been called.

Args:
    data (toast.Data):  The distributed data.
    detectors (list):  A list of detector names or indices.  If None, this
        indicates a list of all detectors.

Returns:
    (value):  None or an Operator-dependent result.
File:      /shared_home/software/soconda_builds/soconda_20260813_0.2.5/lib/python3.13/site-packages/toast/ops/operator.py
Type:      function

In [ ]:
# Customize some traits that are not in the configs, and which are usually
# set in the batch scripts.
job_ops.load.context_file = context_file
job_ops.load.telescope_name = tele_name
job_ops.load.dets_select = det_select
job_ops.load.observations = [obs_id]
job_ops.load.preprocess_config = site_preproc

job_ops.sim_sky_alm.file = "sky_alm.fits"

job_ops.mapmaker.area = area_file
job_ops.mapmaker.nmat_dir = os.path.join(out_dir, "nmat")
job_ops.mapmaker.maxerr = 1.0e-8

In [22]:
print(job_ops.__dict__.keys())

dict_keys(['load', 'det_pointing_radec', 'det_pointing_azel', 'extend_invalid', 'az_intervals', 'bias_cuts', 'raw_jumps', 'trend_cuts', 'raw_deglitch', 'cal_to_pw', 'readout_filter', 'deconvolve_timeconstant', 'pre_proc', 'diff_noise', 'diff_noise_cut', 'noise_cut', 'weights_radec', 'weights_azel', 'pixels_healpix_radec', 'pixels_car_radec', 'det_pointing_radec_sim', 'det_pointing_azel_sim', 'weights_radec_sim', 'weights_azel_sim', 'pixels_healpix_radec_sim', 'pixels_car_radec_sim', 'sim_sky_alm', 'sim_noise', 'sim_atmosphere', 'sim_atmosphere_coarse', 'clear_detdata', 'remove_dc', 'fill_gaps', 'simulate', 'mapmaker'])


## Create Data

Create the starting toast data container and load observations from the context.

In [ ]:
comm = toast.mpi.Comm(world=world)
data = toast.Data(comm=comm)

In [ ]:
job_ops.load.apply(data)

In [ ]:
first_ob = data.obs[0]

In [ ]:
# Plot dets before processing
if rank == 0:
    plot_obs_dets(first_ob, d_start=0, d_end=10, s_end=200000)

## Low-Level Preprocessing

Next we run low-level preprocessing (defined in the config files above).  This works with data prior to any HWPSS filtering.

In [ ]:
job_ops.pre_proc.apply(data)

In [ ]:
job_ops.noise_cut.apply(data)

In [ ]:
# Remove a DC level in each detector for plotting
for det in first_ob.local_detectors:
    good = (first_ob.detdata["flags"][det] & defaults.det_mask_nonscience) == 0
    dc = np.mean(first_ob.detdata["signal"][det][good])
    first_ob.detdata["signal"][det] -= dc

In [ ]:
if rank == 0:
    plot_obs_dets(first_ob, d_start=0, d_end=10, s_start=5000, s_end=200000)
    plot_obs_dets(first_ob, d_start=0, d_end=10, s_start=5000, s_end=40000)

## Simulation

Clear the data and run the simulation operators (defined in the `simulation.yml` config in the current directory).

In [23]:
job_ops.simulate.apply(data)

NameError: name 'data' is not defined

In [ ]:
if rank == 0:
    plot_obs_dets(first_ob, d_start=0, d_end=10, s_start=5000, s_end=200000)
    plot_obs_dets(first_ob, d_start=0, d_end=10, s_start=5000, s_end=40000)

### Notes

The weather model loaded along with the real data is the "typical weather" given by site information from sotodlib.  A future update would be to create a more detailed weather model if the optional housekeeping data was loaded along with the real data.

## Mapmaking

See the config file for the various options used.  Rather than use a large sky area for this small test, we use some toast tools to compute the projection bounds from the data and save that to a footprint file.

In [ ]:
# Cleanup any stale files
for old in glob.glob("mapmaker_pass*"):
    os.remove(old)

In [ ]:
job_ops.mapmaker.apply(data)

## Results

We are only making a single wafer-observation map, so there is not much to see.

In [ ]:
if rank == 0:
    from IPython.display import display, Image
    # hit_root = os.path.join(out_dir, "mapmaker_pass2_sky_hits")
    # map_root = os.path.join(out_dir, "mapmaker_pass2_sky_map")
    hit_root = "mapmaker_pass2_sky_hits"
    map_root = "mapmaker_pass2_sky_map"
    hit_file = f"{hit_root}.fits"
    map_file = f"{map_root}.fits"
    
    toast.vis.plot_wcs_maps(
        #hitfile=hit_file,
        mapfile=map_file,
        format="png",
        cmap="bwr",
        range_I=(-0.0001,0.0001),
        range_Q=(-0.0001,0.0001),
        range_U=(-0.0001,0.0001),
    )
    
    # hit_png = f"{hit_root}.png"
    # display(Image(hit_png))
    map_I_png = f"{map_root}_I.png"
    display(Image(map_I_png))
    map_Q_png = f"{map_root}_Q.png"
    display(Image(map_Q_png))
    map_U_png = f"{map_root}_U.png"
    display(Image(map_U_png))